In [ ]:
DATA_YAML = "config/data.yaml"
MODEL_PATH = "weights/best.pt"
PARAMS_JSON = "config/params.json"

In [ ]:
import yaml
import os

def load_yolo_data(yaml_path):
    # Check if file exists
    if not os.path.exists(yaml_path):
        print(f"File not found: {yaml_path}")
        return None

    with open(yaml_path, 'r', encoding='utf-8') as f:
        try:
            # Use safe_load to avoid executing malicious code, which is standard practice for handling YAML
            data = yaml.safe_load(f)
            return data
        except yaml.YAMLError as exc:
            print(f"Error reading YAML: {exc}")
            return None

config = load_yolo_data(DATA_YAML)

if config:
    print("--- Dataset configuration loaded successfully ---")
    
    # 1. Get number of classes
    NUM_CLASSES = config.get('nc', 0)
    print(f"Number of classes (nc): {NUM_CLASSES}")
    
    # Get DATASET path
    DATASET_DIR = config.get('path', '')

    # 2. Get class name list
    CLASS_NAMES = config.get('names', [])
    print(f"Class names: {CLASS_NAMES}")
    
    # 3. Get train/val paths
    TRAIN_PATH = config.get('train', 'not set')
    VAL_PATH = config.get('val', 'not set')

    TRAIN_IMG_DIR = os.path.join(DATASET_DIR, TRAIN_PATH)
    VAL_IMG_DIR   = os.path.join(DATASET_DIR, VAL_PATH)

    print(f"Dataset path: {DATASET_DIR}")
    print(f"Training set path: {TRAIN_IMG_DIR}")
    print(f"Validation set path: {VAL_IMG_DIR}")

    # If names is in dict format {0: 'person', 1: 'dog'}, handle it like this
    if isinstance(CLASS_NAMES, dict):
        class_list = list(CLASS_NAMES.values())
        print(f"Converted class list: {class_list}")

In [ ]:
import subprocess
import sys

def run_cmd(cmd:list):
    '''
    Ex: 
        cmd = [
        sys.executable, "quant_script/quant_custom.py",
        "--model_path",   "weights/best.pt",
        "--data_dir", "/home/jianhua/Desktop/dataset/SeaDronesSee_MOT",
        "--quant_mode", "calib",
        "--input_size", "640",
        "--num_classes", "4",
        "--extra_path", ".",
        ]
    '''
    
    with subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    ) as proc:
        last_line = ""
        for line in proc.stdout:
            line = line.rstrip("\n")
            # 內容一樣就跳過
            if line == last_line:
                continue
            # 進度條行：用 \r 覆寫同一行 (tqdm 套件特殊格式)
            if "it/s" in line or "s/it" in line:
                sys.stdout.write("\r" + line)
                sys.stdout.flush()
            # 一般訊息：換行輸出
            else:
                sys.stdout.write("\n" + line + "\n")
                sys.stdout.flush()
            last_line = line

## Calib (校準)

In [ ]:
"""
# Step 1 — 校準
python quant_script/quant_custom.py   
    --model_path weights/best.pt   \
    --data_dir /home/jianhua/Desktop/dataset/SeaDronesSee_MOT    \
    --num_classes 4, \
    --quant_mode calib   \
    --input_size 640 \
    --extra_path .
"""

cmd = [
    sys.executable, "quant_script/quant_custom.py",
    "--model_path",   MODEL_PATH,
    "--data_dir", DATASET_DIR,
    "--num_classes", str(NUM_CLASSES),
    "--quant_mode", "calib",
    "--input_size", "640",
    "--extra_path", ".",
]

run_cmd(cmd=cmd)

## Eval mAP (test)

In [ ]:
'''
# Step 2 — 評估 mAP（test）
python quant_script/quant_custom.py \
  --model_path weights/best.pt \
  --data_dir   /path/to/dataset \
  --params, config/params.json \
  --num_classes 4 \
  --quant_mode test \
  --input_size 640 \
  --extra_path . 
'''

cmd = [
    sys.executable, "quant_script/quant_custom.py",
    "--model_path", MODEL_PATH,
    "--data_dir", DATASET_DIR,
    "--params", PARAMS_JSON,
    "--num_classes", str(NUM_CLASSES),
    "--quant_mode", "test",
    "--input_size", "640",
    "--extra_path", ".",
]

run_cmd(cmd=cmd)

## Export xmodel

In [ ]:
'''
# Step 3 — 匯出部署檔
python quant_script/quant_custom.py \
  --model_path weights/best.pt \
  --data_dir  /home/jianhua/Desktop/dataset/SeaDronesSee_MOT  \
  --params, config/params.json \
  --num_classes 4 \
  --quant_mode test \
  --input_size 640 \
  --extra_path . \
  --target DPUCZDX8G_ISA1_B4096 \
  --batch_size 1 \
  --subset_len 1 \
  --deploy
'''

cmd = [
    sys.executable, "quant_script/quant_custom.py",
    "--model_path",   MODEL_PATH,
    "--data_dir", DATASET_DIR,
    "--params", PARAMS_JSON,
    "--num_classes", str(NUM_CLASSES),
    "--quant_mode", "test",
    "--input_size", "640",
    "--extra_path", ".",
    "--target", "DPUCZDX8G_ISA1_B4096",
    "--batch_size", "1",
    "--subset_len", "1",
    "--deploy",
]

run_cmd(cmd=cmd)

In [22]:
import sys
import os

# 從目前 Python 路徑逆推 conda 環境根目錄
# /home/jianhua/miniconda3/envs/vai_pytorch/bin/python -> /home/jianhua/miniconda3/envs/vai_pytorch
CONDA_ENV = os.path.dirname(os.path.dirname(sys.executable))
print(f"Conda env path : {CONDA_ENV}")

Conda env path : /home/jianhua/miniconda3/envs/vai_pytorch


In [ ]:
# Pytorch xmodel 轉 DPU xmodel 工具
VAI_C_XIR = os.path.join(CONDA_ENV, "bin", "vai_c_xir")
XMODEL_PATH = "quantize_result/YOLO_int.xmodel"
ARCH_FILE = ""

print(f"vai_c_xir tool path: {VAI_C_XIR}")
print(f"xmodel path: {XMODEL_PATH}")

cmd = [VAI_C_XIR, XMODEL_PATH]

vai_c_xir tool path: /home/jianhua/miniconda3/envs/vai_pytorch/bin/vai_c_xir
xmodel path: quantize_result/YOLO_int.xmodel


In [19]:
XMODEL_PATH = "quantize_result/YOLO_int.xmodel"
cmd = [sys.executable, "quant_script/view_xmodel.py", XMODEL_PATH]
run_cmd(cmd)



  Vitis-AI xmodel Inspector  (PyXIR / xir)


  Loading: quantize_result/YOLO_int.xmodel

  Loaded successfully.




  Graph Summary


  Graph name  : YOLO

  Total ops   : 641



  Op type distribution (11 unique types):

    fix                                      x 264

    const                                    x 168

    conv2d                                   x 78

    relu                                     x 77

    concat                                   x 20

    add                                      x 12

    strided_slice                            x 10

    depthwise-conv2d                         x 6

    maxpool2d                                x 3

    resize                                   x 2

    data                                     x 1




  Graph Input / Output Tensors




  Input tensors (0):



  Output tensors (3):

  Could not retrieve IO tensors: 'xir.Tensor' object has no attribute 'get_name'




  Subgraph Structure


> root

  op_num=641  ch